## Imports

In [1]:
import numpy as np
import pandas as pd

import math
import random

## Loading
Loading the data, perhaps via chunking

Note: Could cause issues with duplicates later...

In [2]:
"""
Loads chunks of data from csv file

Returns:
    pd.DataFrame: The selected chunk of data
"""
def load_csv_chunk(
    filepath: str,              # Path to .csv file
    chunks: int = 1,            # Number of (roughly) equal parts to split the csv into
    chunk_idx: int = 0,         # Index of the chunk to load (zero-based)
    offset: int = 1,            # Number of header lines in file
    **read_csv_kwargs           # Extra arguments passed to pd.read_csv()
):

    # Count number of data rows, excluding the header
    #   Get specified encoding, otherwise default utf-8
    with open(filepath, "r", encoding=read_csv_kwargs.get("encoding", "utf-8")) as f:
        total_rows = sum(1 for line in f) - offset

    # Chunking/Slicing Information
    chunk_size = math.ceil(total_rows / chunks)           # Size of chunk
    start_row = chunk_idx * chunk_size                     # Index of start row in actual data
    end_row = min(start_row + chunk_size, total_rows)     # Index of end row in actual data
    nrows = end_row - start_row                           # Number of rows to read
    header_row = offset - 1                               # Row index for the header information (assumed as the last header row)

    # Extract the header/columns
    columns = pd.read_csv(
        filepath,
        skiprows=header_row,
        nrows=0,
        **read_csv_kwargs
    ).columns

    # Read only the selected chunk of data
    df = pd.read_csv(
        filepath,
        skiprows=offset + start_row,
        nrows=nrows,
        names=columns,
        header=None,
        **read_csv_kwargs
    )

    return df

In [3]:
raw_sample = load_csv_chunk("UK-Sanctions-List.csv", offset=2)

C:\Users\alecz\AppData\Local\Temp\ipykernel_28508\4224360936.py:36: DtypeWarning: Columns (0: IMO number, 1: Current owner/operator (s), 2: Previous owner/operator (s), 3: Current believed flag of ship, 4: Previous flags, 5: Type of ship) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


## Exploration
Explore the fields (columns), look for data we're interested in (Name, DOB, Countries, other...)

By the end of this step, we should have an understanding of which fields are useful and which can be excluded.

In [4]:
"""
Gives useful information and examples of data in a specific column
Helps decide if it is useful for our purposes
"""
def profile_column(
        df: pd.DataFrame,        # Dataframe to analyse
        col_name: str,           # Column identifier
        top_n: int = 5,          # Show top_n most common values
        sampl_n: int = 10,       # Show sampl_n number of random examples
) -> None:
    
    col = df[col_name]
    rows = len(col)
    missg = col.isna().sum()
    non_missg = col.notna().sum()
    missg_pct = round((missg / rows) * 100, 2)
    unique = col.nunique(dropna=True)

    # Basic Info
    print("#" * 40)
    print(f"Column: {col_name}")
    print("#" * 40)
    print(f"Total rows:          {rows}")
    print(f"Data type:           {col.dtype}")
    print(f"Non-missing values:  {non_missg}")
    print(f"Missing values:      {missg}")
    print(f"Missing %:           {missg_pct}%")
    print(f"Unique values:       {unique}")
    if non_missg > 0:
        print(f"Most common value:   {col.value_counts(dropna=True).index[0]}")
        print(f"Most common count:   {col.value_counts(dropna=True).iloc[0]}")

    # Print top occurences
    print("\nTop value counts:")
    print("-" * 20)
    print(col.value_counts(dropna=False).head(top_n))

    # Print some random examples
    print("\nRandom non-missing examples:")
    print("-" * 20)
    examples = col.dropna().drop_duplicates()
    if len(examples) == 0:
        print("No non-missing examples available.")
    else:
        print(
            examples
            .sample(min(sampl_n, len(examples)))
            .to_string(index=False)
        )


    # Print longest example (if text area)
    print("\nLongest non-missing examples:")
    print("-" * 20)
    if len(examples) == 0:
        print("No non-missing examples available.")
    else:
        longest_examples = (
            examples
            .astype(str)
            .sort_values(key=lambda x: x.str.len(), ascending=False)
            .head(sampl_n)
        )
        print(longest_examples.to_string(index=False))


In [5]:
# Other Basic Tools
# df.shape, df.head()
# df.columns
# df.dtypes, df.info()
# df.isna().sum(), df.count().sort_values(ascending=True)
# df.nunique(), df["some_column"].value_counts(dropna=False)

In [6]:
raw_sample.head()

,Last Updated,Unique ID,OFSI Group ID,UN Reference Number,Name 6,Name 1,Name 2,Name 3,Name 4,Name 5,...,IMO number,Current owner/operator (s),Previous owner/operator (s),Current believed flag of ship,Previous flags,Type of ship,Tonnage of ship,Length of ship,Year Built,Hull identification number (HIN)
0,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
raw_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 57033 entries, 0 to 57032
Data columns (total 58 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Last Updated                                57033 non-null  str    
 1   Unique ID                                   57033 non-null  str    
 2   OFSI Group ID                               54769 non-null  float64
 3   UN Reference Number                         20532 non-null  str    
 4   Name 6                                      56921 non-null  str    
 5   Name 1                                      23893 non-null  str    
 6   Name 2                                      12107 non-null  str    
 7   Name 3                                      2573 non-null   str    
 8   Name 4                                      477 non-null    str    
 9   Name 5                                      101 non-null    str    
 10  Name type            

In [8]:
profile_column(raw_sample, "Subsidiaries", top_n=10)

########################################
Column: Subsidiaries
########################################
Total rows:          57033
Data type:           str
Non-missing values:  5595
Missing values:      51438
Missing %:           90.19%
Unique values:       206
Most common value:   AIS Iran Co
Most common count:   420

Top value counts:
--------------------
Subsidiaries
NaN                                                51438
AIS Iran Co                                          420
Electronic Component Industries (ECI)                420
Iranian Electronic Science & Research Institute      420
Iran Electronics Industries Co (Saga)                420
Isfahan Optics Industry (SAPA)                       420
Security Industry Information Space (SASTOBA)        420
Shiraz Electronics Industries (Sara Shiraz)          420
Telecommunication Industries of Iran (SAMA)          420
The Institute of Isayran Co                          420
Name: count, dtype: int64

Random non-missing examples:
--

In [10]:
"""
Other experiments
"""

# Checking pipelining in Subsidiaries and Parent company
# filt = raw_sample[
#     (raw_sample["Subsidiaries"].str.contains(";"))
# ]
# filt["Subsidiaries"]


# Checking entity specific information, ie. ships don't have passport number etc...
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Phone number", "Website", 
#               "Email address", "National Identifier number", 
#               "Passport number", "Business registration number (s)", "IMO number"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# Checking only individuals have DOB and Gender
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["D.O.B", "Gender"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# Checking Country information per entity type
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Nationality(/ies)", "Country of birth", "Current believed flag of ship", "Previous flags"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")
    
# Making sure only entities have subsidiaries or parent companies
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Subsidiaries", "Parent company"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")

# Making sure only entities have subsidiaries or parent companies
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Title"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# Seeing name relationships across entities (drop these later for space)
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Name type", "Alias strength", "Name non-latin script", "Non-latin script type", "Non-latin script language"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# filt = raw_sample[
#     (raw_sample["Designation Type"] == "Individual") &
#     (raw_sample["Name 6"].isna())
# ]
# filt.iloc[random.randint(0, len(filt)-1)]


# Check the relationship between designation type and type of entity (does it apply to businesses only?)
# filt = raw_sample[
#     (raw_sample["Designation Type"] == "Entity") &
#     (raw_sample["Type of entity"].notna())
# ]
# len(filt)



# Seeing how many NaNs for alias strength there are even if the name type is an alias
# filt = raw_sample[
#     ((raw_sample["Name type"] == "Alias") | (raw_sample["Name type"] == "ALias")) &
#     (raw_sample["Alias strength"].notna())
# ]
# len(filt)


# filt = raw_sample[
#     (raw_sample["Name type"].isna())
# ]
# filt.iloc[0]


# Date Designated (most recent)
# pd.to_datetime(raw_sample["Date Designated"]).max()


# Address Format checking
# filt = raw_sample[
#     (raw_sample["Address Line 1"].str.contains("TSUEN WAN, NEW TERRITORIES, HONG KONG,UNIT 601", na=False))
# ]
# filt.iloc[0]

# Country Info Checking
# filt = raw_sample[
#     (raw_sample["Town of birth"].notna()) &
#     (raw_sample["Country of birth"].isna()) &
#     (raw_sample["Address Country"].isna()) &
#     (raw_sample["Nationality(/ies)"].isna())
# ]

# ID Differentiation
# filt = raw_sample[
#     (raw_sample["UN Reference Number"] == "TAi.004") 
# ]
# filt["Unique ID"].value_counts()

# Spotting Associations across fields
# filt[["Regime Name", "Designation Type", "Designation source", "Address Country", "Nationality(/ies)", "Country of birth"]].sample(n=20, replace=True)# ["Regime Name"].iloc[0]

# Non-latin script type and Non-latin script language
# filt = raw_sample[
#     (raw_sample["Name non-latin script"].notna()) &
#     (raw_sample["Non-latin script type"].isna()) 
# ]
# filt.iloc[0]


'\nOther experiments\n'

### Findings

Guiding the data cleaning process.Metadata found here: https://www.gov.uk/guidance/format-guide-for-the-uk-sanctions-list

The relevant columns that will contribute to building the transformed data source directly:
   - <span style="color: #61E283;">**Unique ID**</span>: Leaving this in as a reference/foreign key to the original sanctions dataset could help validate their presence in the original dataset later on. Not useful for matching, but has no missing values and could help keep a relationship with our transformed dataset.
   - <span style="color: #61E283;">**Name 1-6**</span>: Very important name information. Name 6 is the surname or the full name of the entity/ship. Names 1-5 are first and middle names, with increasing missing values as expected for rarer longer names
   - <span style="color: #61E283;">**Name type**</span>: Potentially helpful information if aliases are matched with innocent parties, this can be used with alias strength to determine a matching confidence score
   - <span style="color: #61E283;">**Alias strength**</span>: as above
   - <span style="color: #61E283;">**Name non-latin script**</span>: certainly useful for those few percent that may write their names in a different alphabet, helps make the matching system more robust
   - <span style="color: #61E283;">**Regime Name**</span>: not useful for matching but gives the bank context on which sanctions regime applies to the entity, it is also dense with no missing values
   - <span style="color: #61E283;">**Designation Type**</span>: provides context about the name, are they an individual, entity or ship (not directly useful for matching but can contribute to match validation/confidence)
   - <span style="color: #61E283;">**Sanctions Imposed**</span>: provides context about the type of sanctions the matched entity may face (again, not useful directly for matching, but may guide the bank's procedure after matching)
   - <span style="color: #61E283;">**Other Information**</span>: provides context about the person and their sanctions which isn't useful directly for matching, but can help inform later decisions. It may also be used to determine associated countries
   - <span style="color: #61E283;">**UK Statement of Reasons**</span>: as above, provides more context that could be used to further clarify situation after matching, not necessarily useful for the actual matching
   - <span style="color: #61E283;">**Type of entity**</span>: as above, provides more context that could be used to further clarify situation after matching, not necessarily useful for the actual matching (but for business/entity organisations rather than individuals)
   - <span style="color: #61E283;">**Address 1-6, Postal Code, Country**</span>: could use the address to match customer records or further improve matching confidence score. (also find associated countries)
   - <span style="color: #61E283;">**Phone number, Website, Email address, National Identifier number, Passport number, Business registration number, IMO number**</span>: further information that can guide matching confidence or be used as secondary matching criteria
   - <span style="color: #61E283;">**D.O.B, Gender**</span>: futher validation for name matching (eg. two people with same name but different birthdays, one is innocent)
   - <span style="color: #61E283;">**Nationality(/ies), Country of birth, Current believed flag of ship, previous flags**</span>: useful associated countries information
   - <span style="color: #61E283;">**Subsidiaries, Parent Company**</span>: the matching system likely needs to flag these too, treated as separate entity names for example (eg. companies under parent company being sanctioned)
   - <span style="color: #61E283;">**Current owner, Previous owner**</span>: similar to above but for ships
   - <span style="color: #61E283;">**Type of ship, Tonnage of ship, Length, Year built**</span>: validate match confidence but for ships



The columns we decided have no use for us are:
   - <span style="color: #F76262;">**Last Updated**</span>: If an entity already appears in this list, we would want the system to match them. There is no date of release of sanction that could be used alongside this field to clarify the sanctions validity.
   - <span style="color: #F76262;">**Date Designated**</span>: for similar reasons to the above
   - <span style="color: #F76262;">**OFSI Group ID**</span>: It was thought this could be used as a compound key with the Unique ID, but after exploring further, it held no differentiating power (legacy ID)
   - <span style="color: #F76262;">**UN Reference Number**</span>: similar to the above, it was mostly missing and held no differentiating power
   - <span style="color: #F76262;">**Designation source**</span>: it is already assumed we are curating a dataset for UK sanctions, and this field only differentiates between UN or UK which both matter (redundant)
   - <span style="color: #F76262;">**HIN**</span>: NaN column, can be discarded
   

These fields may be helpful, for example for imputation, but will ultimately not be needed in our transformed dataset:
   - <span style="color: #d48748;">**Title**</span>: some titles can be very distinguishing (Second Vice-President of the National Consti...), others may be very general (Captain, General...), shouldn't reliably be used for matching (or even match checking), titles may change, its mostly missing values, but maybe can impute country information or things like that??
   - <span style="color: #d48748;">**Position**</span>: similar to above
   - <span style="color: #d48748;">**Non-latin script type**</span>: not helpful for name matching purposes, but may help discern country information??
   - <span style="color: #d48748;">**Non-latin script language**</span>: as above
   - <span style="color: #d48748;">**National Identifier additional information, Passport additional information**</span>: not useful for matching, does provide context but not highly relevant. it may be used for identifying associated countries from the text though.
   - <span style="color: #d48748;">**Town of birth**</span>: can be used to impute country association data

## Designing
This is where we can start to formulate what our database may look like, consolidating our exploration ideas before data cleaning/transformation ensues

The proposed solution involves cleaning and normalising the dataset into 4 component datasets:
1. <span style="color: #61E283;">**name_index.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: newly created Sanctioned Entity ID, this can function as our new primary key, and foreign key across the datasets
    - <span style="color: #5d8eb8;">Full Name</span>: full name field, including any middle names and surname
    - <span style="color: #5d8eb8;">Designation Type</span>: identifies whether the party is an individual, entity or a ship
2. <span style="color: #61E283;">**individuals.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">LUID</span>: Legacy Unique ID, keeps the relationship to the original gov.uk dataset
    - <span style="color: #5d8eb8;">Surname</span>: from Name 6
    - <span style="color: #5d8eb8;">Given Names</span>: Name 1+2+...+5
    - <span style="color: #5d8eb8;">Name (Non-Latin Script)</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Type</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Language</span>: =
    - <span style="color: #5d8eb8;">Name Type</span>: =
    - <span style="color: #5d8eb8;">Alias Strength</span>: =
    - <span style="color: #5d8eb8;">D.O.B</span>: =
    - <span style="color: #5d8eb8;">Gender</span>: =
    - <span style="color: #5d8eb8;">Title</span>: =
    - <span style="color: #5d8eb8;">Position</span>: =
    - <span style="color: #5d8eb8;">Nationalities</span>: =
    - <span style="color: #5d8eb8;">Birth Country</span>: =
    - <span style="color: #5d8eb8;">Birth Town</span>: =
    - <span style="color: #5d8eb8;">Address Lines</span>: Address Lines 1+2+...+6
    - <span style="color: #5d8eb8;">Address Postal Code</span>: =
    - <span style="color: #5d8eb8;">Address Country</span>: =
    - <span style="color: #5d8eb8;">Phone Number</span>: =
    - <span style="color: #5d8eb8;">Website</span>: =
    - <span style="color: #5d8eb8;">Email</span>: =
    - <span style="color: #5d8eb8;">National Identifier Number</span>: =
    - <span style="color: #5d8eb8;">National Identifier Info</span>: =
    - <span style="color: #5d8eb8;">Passport Number</span>: =
    - <span style="color: #5d8eb8;">Passport Info</span>: =

3. <span style="color: #61E283;">**entities.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">LUID</span>: ^^
    - <span style="color: #5d8eb8;">Name</span>: Usually just Name 6, but concat 1-6
    - <span style="color: #5d8eb8;">Name (Non-Latin Script)</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Type</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Language</span>: =
    - <span style="color: #5d8eb8;">Name Type</span>: =
    - <span style="color: #5d8eb8;">Alias Strength</span>: =
    - <span style="color: #5d8eb8;">Address Lines</span>: Address Lines 1+2+...+6
    - <span style="color: #5d8eb8;">Address Postal Code</span>: =
    - <span style="color: #5d8eb8;">Address Country</span>: =
    - <span style="color: #5d8eb8;">Phone Number</span>: =
    - <span style="color: #5d8eb8;">Website</span>: =
    - <span style="color: #5d8eb8;">Email</span>: =
    - <span style="color: #5d8eb8;">Business Reg</span>: =
    - <span style="color: #5d8eb8;">Type</span>: from type of entity
    - <span style="color: #5d8eb8;">Subsidiaries</span>: =
    - <span style="color: #5d8eb8;">Parent Company</span>: =
    
4. <span style="color: #61E283;">**ships.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">LUID</span>: ^^
    - <span style="color: #5d8eb8;">Name</span>: Usually just Name 6, but concat 1-6
    - <span style="color: #5d8eb8;">Name (Non-Latin Script)</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Type</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Language</span>: =
    - <span style="color: #5d8eb8;">Name Type</span>: =
    - <span style="color: #5d8eb8;">Alias Strength</span>: =
    - <span style="color: #5d8eb8;">IMO Number</span>: =
    - <span style="color: #5d8eb8;">Current Believed Flag</span>: =
    - <span style="color: #5d8eb8;">Previous Flags</span>: =
    - <span style="color: #5d8eb8;">Type</span>: from type of ship
    - <span style="color: #5d8eb8;">Tonnage</span>: =
    - <span style="color: #5d8eb8;">Length</span>: =
    - <span style="color: #5d8eb8;">Year Built</span>: =


5. <span style="color: #61E283;">**sanctions.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">Regime Name</span>: =
    - <span style="color: #5d8eb8;">Sanctions Imposed</span>: depipeline this ideally
    - <span style="color: #5d8eb8;">Other Info</span>: =
    - <span style="color: #5d8eb8;">UK Statement of Reasons</span>: =
    - <span style="color: #5d8eb8;">Date Designated</span>: =
    - <span style="color: #5d8eb8;">Last Updated</span>: =

The motivating idea behind this scheme is that it is very likely that the customer records will include a name or a designation type. Here is the typical algorithm flow:

1. Designation Type is known
    - Perform the matching algorithm on the narrower datasets (either: individuals, entities or ships .csv) with whatever fields you have

2. Otherwise, Name is known
    - Use name_index.csv to find candidate SEID foreign keys and their designation types, and check those corresponding csv and keys further for matching validity/confidence etc...
    - SEID can be numeric and datasets can be sorted by SEID to allow for binary search?

3. Otherwise, Designation Type and Name is not known (e.g. only have an address)
    - Look at the csv headers for clues about the designation type (e.g. only ships have IMO numbers, only individuals have Passport numbers, etc...)
    - Otherwise if limited matching criteria to go off of, scan for matches across all 3 datasets (worst case scenario)

Note: name_index could be expanded with more fields, but there aren't other useful shared fields between all three entities


## Cleaning

Handle the cleaning for the relevant data fields that we want our transformed data set to have

In [10]:
not_used = [
    "Last Updated",
    "Date Designated",
    "OFSI Group ID",
    "UN Reference Number",
    "Designation source",
    "Hull identification number (HIN)"
]
raw_sample.drop(not_used, axis=1, inplace=True)

In [11]:
#len(raw_sample.columns)

### 1. Name

Relevant fields: Name 1-6, Name non-latin script, Subsidiaries, Parent Company, Current/Previous owner\
Observations:\
    - Lots of NaNs from later Name fields (longer names less likely)\
    - Some individuals have their full name written in name 1, rather than formatting the surname in name 6\
    - Some people only have a non-latin script name, and vice versa\
    - Varying case script\

In [12]:
# filt = raw_sample[
#     (raw_sample["Name non-latin script"].notna()) &
#     (raw_sample["Name 1"].isna()) &
#     (raw_sample["Name 6"].isna())
# ]
# filt.iloc[0]

In [13]:
"""
Merge Names 1-6 fields into a full name
"""
raw_sample["Full Name"] = (
    raw_sample[[                              # Ordered Name columns 1-6
        "Name 1",
        "Name 2",
        "Name 3",
        "Name 4",
        "Name 5",
        "Name 6"
    ]]
    .fillna("")                               # Replace NaN with empty string
    .agg(" ".join, axis=1)                    # Joins name fields with space in between (across columns)
    .str.replace(r"\s+", " ", regex=True)     # Cleans up extra whitespace characters (replace them with just 1 space)
    .str.strip()                              # Remove any left/right trailing spaces
    .str.lower()                              # Standardise names to lower case (names are case insensitive)
)

In [30]:
# profile_column(raw_sample, "Full Name", top_n=20)
# filt = raw_sample[
#     (raw_sample["Full Name"].str.contains(";"))
# ]
# filt["Full Name"][1]

In [ ]:
"""
WE ARE GOING TO HAVE TO REMOVE EMPTY NAMES, "", AT THE END IF NO NAME INFO COULD BE PROVIDED FOR THE RECORD
"""

## Duplicates

Typically, check the clean transformed data for any duplicates

## QC

Quality control, check if theres any missing values, any obvious duplicates, the range/values fields take, etc...

## Export